# Jarvis LLM v1 — Google Colab training notebook

**From-scratch, small decoder-only Transformer.** This notebook is deliberately conservative:
- Starts with a ~2M parameter smoke-test model.
- Uses custom BPE tokenizer (trained on your own corpus).
- Uses mixed precision only when supported.
- Uses gradient accumulation, small micro-batches, gradient clipping.
- Saves checkpoints and tokenizer to Google Drive.
- Can resume from the latest checkpoint.
- Does not assume a particular Colab GPU.

> Important: this is an experimental small language model, not a ChatGPT-scale model.
> 500 MB of JSON does not guarantee 500 MB of clean conversational text. Model quality depends
> heavily on the actual corpus and its diversity.

## Step 0 — Choose runtime

In Colab: **Runtime → Change runtime type → GPU** if available.
Run the next cell before training. If Colab assigns no GPU, the notebook will still work in CPU mode,
but training will be much slower.

In [ ]:
import os, sys, json, math, time, random, shutil, subprocess, platform
import torch

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    props = torch.cuda.get_device_properties(0)
    print("VRAM GiB:", round(props.total_memory / 1024**3, 2))
print("System RAM GiB:", round(__import__("psutil").virtual_memory().total / 1024**3, 2))

## Step 1 — Mount Google Drive and create persistent folders

Upload your 10 JSON files into a folder in Drive, for example `MyDrive/jarvis_data/`.
Set `DATA_DIR` below to that folder. Checkpoints are saved separately in `MyDrive/JarvisLLM/`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DATA_DIR = "/content/drive/MyDrive/jarvis_data"   # change if your folder differs
WORK_DIR = "/content/jarvis_work"                 # fast temporary disk
DRIVE_OUT = "/content/drive/MyDrive/JarvisLLM"
os.makedirs(WORK_DIR, exist_ok=True)
os.makedirs(DRIVE_OUT, exist_ok=True)
os.makedirs(f"{WORK_DIR}/raw", exist_ok=True)
os.makedirs(f"{WORK_DIR}/processed", exist_ok=True)
os.makedirs(f"{DRIVE_OUT}/checkpoints", exist_ok=True)
print("Data folder exists:", os.path.isdir(DATA_DIR))
print("Files:", os.listdir(DATA_DIR)[:20] if os.path.isdir(DATA_DIR) else "NOT FOUND")

## Step 2 — Install only the required packages

In [ ]:
!pip -q install tokenizers tqdm psutil

## Step 3 — Copy JSON files from Drive to Colab's temporary disk

Training from `/content` is usually faster than reading many small files from Drive.
Drive remains the persistent source and destination.

In [ ]:
from pathlib import Path
src = Path(DATA_DIR)
raw = Path(WORK_DIR) / "raw"
json_files = list(src.rglob("*.json"))
if not json_files:
    raise FileNotFoundError(f"No .json files found under {DATA_DIR}")
for f in json_files:
    rel = f.relative_to(src)
    dest = raw / rel
    dest.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(f, dest)
print("Copied JSON files:", len(json_files))
print("Total raw bytes:", sum(f.stat().st_size for f in raw.rglob("*.json")))

## Step 4 — Parse common conversation JSON formats

Supported:
- `messages: [{role, content}]`
- `conversations: [...]`
- `question` + `answer`
- `text`

If your JSON schema is different, stop here and share 2–3 sample records so the parser can be adapted.
We split by complete records, deduplicate exact duplicates, and keep a validation set.

In [ ]:
import json, random, re
from pathlib import Path

SPECIAL = ["<|pad|>", "<|unk|>", "<|bos|>", "<|eos|>",
           "<|system|>", "<|user|>", "<|assistant|>"]
ROLES = {"system", "user", "assistant"}

def role_norm(x):
    x = str(x or "").lower()
    if x in ("human", "prompt", "question"): return "user"
    if x in ("gpt", "bot", "model", "completion"): return "assistant"
    return x if x in ROLES else None

def content_text(x):
    if isinstance(x, str): return x.strip()
    if isinstance(x, list):
        out=[]
        for a in x:
            if isinstance(a, str): out.append(a)
            elif isinstance(a, dict) and a.get("type")=="text": out.append(str(a.get("text","")))
        return "\n".join(out).strip()
    return ""

def parse_messages(msgs):
    out=[]
    for m in msgs:
        if not isinstance(m, dict): continue
        role=role_norm(m.get("role") or m.get("from") or m.get("speaker"))
        txt=content_text(m.get("content",m.get("value",m.get("text",""))))
        if role and txt: out.append(f"<|{role}|>\n{txt}\n")
    return "<|bos|>\n"+"".join(out)+"<|eos|>" if out else ""

def walk(obj):
    if isinstance(obj, list):
        for x in obj: yield from walk(x)
    elif isinstance(obj, dict):
        if isinstance(obj.get("messages"), list):
            x=parse_messages(obj["messages"])
            if x: yield x
        elif isinstance(obj.get("conversations"), list):
            x=parse_messages(obj["conversations"])
            if x: yield x
            else: yield from walk(obj["conversations"])
        elif "question" in obj and "answer" in obj:
            q,a=content_text(obj["question"]),content_text(obj["answer"])
            if q and a: yield f"<|bos|>\n<|user|>\n{q}\n<|assistant|>\n{a}\n<|eos|>"
        elif "text" in obj:
            t=content_text(obj["text"])
            if t: yield f"<|bos|>\n{t}\n<|eos|>"
        else:
            for v in obj.values():
                if isinstance(v,(dict,list)): yield from walk(v)

docs, seen = [], set()
for fp in Path(WORK_DIR,"raw").rglob("*.json"):
    try:
        obj=json.loads(fp.read_text(encoding="utf-8"))
    except Exception as e:
        print("Skipping invalid JSON:", fp.name, str(e)[:160])
        continue
    for d in walk(obj):
        d=d.strip()
        if len(d)>=12 and d not in seen:
            seen.add(d); docs.append(d)

if len(docs)<20:
    print("WARNING: very few parsed records. Inspect the JSON structure before training.")
random.Random(42).shuffle(docs)
nval=max(1,int(len(docs)*0.02)) if len(docs)>1 else 0
val_docs, train_docs=docs[:nval],docs[nval:]
Path(WORK_DIR,"processed","train.txt").write_text("\n".join(train_docs),encoding="utf-8")
Path(WORK_DIR,"processed","val.txt").write_text("\n".join(val_docs),encoding="utf-8")
stats={"json_files":len(list(Path(WORK_DIR,"raw").rglob("*.json"))),
       "records":len(docs),"train_records":len(train_docs),"val_records":len(val_docs),
       "train_chars":sum(map(len,train_docs)),"val_chars":sum(map(len,val_docs))}
print(json.dumps(stats,indent=2))

## Step 5 — Train your custom BPE tokenizer

We use byte-level BPE with Unicode NFC normalization. The special chat tokens are reserved.
For a first run, keep vocabulary at 8,000. Do not increase it before seeing corpus statistics.

In [ ]:
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders, normalizers

tok = Tokenizer(models.BPE(unk_token="<|unk|>"))
tok.normalizer = normalizers.NFC()
tok.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
tok.decoder = decoders.ByteLevel()
trainer = trainers.BpeTrainer(
    vocab_size=8000, min_frequency=2, special_tokens=SPECIAL,
    initial_alphabet=pre_tokenizers.ByteLevel.alphabet()
)
tok.train([f"{WORK_DIR}/processed/train.txt"], trainer)
TOK_DIR=f"{DRIVE_OUT}/tokenizer"
os.makedirs(TOK_DIR,exist_ok=True)
tok.save(f"{TOK_DIR}/tokenizer.json")
print("Tokenizer vocab:",tok.get_vocab_size())
for s in ["नमस्ते Jarvis", "Hello, how are you?", "तुम कैसे हो bro?"]:
    e=tok.encode(s)
    print(s, "=>", e.ids, "=>", tok.decode(e.ids))

## Step 6 — Select a safe model size based on actual VRAM

The first pass is a **tiny smoke test**. Only after it completes and saves a checkpoint should you
try the larger config. The approximate parameter count below excludes small overheads.

In [ ]:
def model_config_for_device():
    if torch.cuda.is_available():
        vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
    else:
        vram = 0
    # Conservative micro-batch and context. Increase only after stable runs.
    if vram >= 12:
        cfg=dict(n_layer=4,n_head=4,n_embd=128,block_size=256,batch_size=4,grad_accum=8)
    elif vram >= 8:
        cfg=dict(n_layer=4,n_head=4,n_embd=128,block_size=192,batch_size=2,grad_accum=8)
    elif vram >= 4:
        cfg=dict(n_layer=2,n_head=4,n_embd=96,block_size=128,batch_size=1,grad_accum=16)
    else:
        cfg=dict(n_layer=2,n_head=4,n_embd=64,block_size=96,batch_size=1,grad_accum=8)
    cfg.update(vocab_size=tok.get_vocab_size(),dropout=0.1)
    return cfg, vram

CFG, VRAM_GB = model_config_for_device()
print("Detected VRAM GiB:",round(VRAM_GB,2))
print("Selected config:",CFG)

## Step 7 — Build the small decoder-only Transformer

This is a simple pre-LayerNorm GPT-style model with causal attention and tied input/output embeddings.
The first model is intentionally small to validate data, loss, saving and resuming.

In [ ]:
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

class CausalBlock(nn.Module):
    def __init__(self,cfg):
        super().__init__()
        d,h=cfg["n_embd"],cfg["n_head"]
        self.h=h; self.dh=d//h
        self.ln1=nn.LayerNorm(d); self.ln2=nn.LayerNorm(d)
        self.qkv=nn.Linear(d,3*d,bias=False); self.proj=nn.Linear(d,d,bias=False)
        self.fc1=nn.Linear(d,4*d); self.fc2=nn.Linear(4*d,d)
        self.drop=nn.Dropout(cfg["dropout"])
    def forward(self,x):
        b,t,c=x.shape
        z=self.ln1(x); q,k,v=self.qkv(z).chunk(3,-1)
        def shape(a): return a.view(b,t,self.h,self.dh).transpose(1,2)
        q,k,v=map(shape,(q,k,v))
        # PyTorch scaled_dot_product_attention uses an efficient causal kernel where available.
        a=F.scaled_dot_product_attention(q,k,v,is_causal=True)
        a=a.transpose(1,2).contiguous().view(b,t,c)
        x=x+self.drop(self.proj(a))
        z=self.ln2(x)
        x=x+self.drop(self.fc2(F.gelu(self.fc1(z))))
        return x

class JarvisGPT(nn.Module):
    def __init__(self,cfg):
        super().__init__(); d=cfg["n_embd"]
        self.cfg=cfg
        self.wte=nn.Embedding(cfg["vocab_size"],d)
        self.wpe=nn.Embedding(cfg["block_size"],d)
        self.drop=nn.Dropout(cfg["dropout"])
        self.blocks=nn.ModuleList([CausalBlock(cfg) for _ in range(cfg["n_layer"])])
        self.ln=nn.LayerNorm(d); self.head=nn.Linear(d,cfg["vocab_size"],bias=False)
        self.head.weight=self.wte.weight
    def forward(self,idx,targets=None):
        b,t=idx.shape
        if t>self.cfg["block_size"]: raise ValueError("Sequence too long")
        pos=torch.arange(t,device=idx.device)
        x=self.drop(self.wte(idx)+self.wpe(pos))
        for block in self.blocks: x=block(x)
        logits=self.head(self.ln(x))
        loss=F.cross_entropy(logits.reshape(-1,logits.size(-1)),targets.reshape(-1)) if targets is not None else None
        return logits,loss

def nparams(m): return sum(p.numel() for p in m.parameters() if p.requires_grad)
model=JarvisGPT(CFG)
print("Parameters:",f"{nparams(model):,}",f"({nparams(model)/1e6:.2f}M)")

## Step 8 — Tokenize corpus and create fixed-length training blocks

We tokenize in this cell and save token IDs locally. If this cell runs out of RAM, stop and process
the corpus in shards; do not repeatedly rerun full tokenization without checking memory.

In [ ]:
import numpy as np
def encode_file(path):
    text=Path(path).read_text(encoding="utf-8")
    return tok.encode(text).ids

train_ids=encode_file(f"{WORK_DIR}/processed/train.txt")
val_ids=encode_file(f"{WORK_DIR}/processed/val.txt")
print("Train tokens:",len(train_ids),"Validation tokens:",len(val_ids))
if len(train_ids) < CFG["block_size"]+2:
    raise RuntimeError("Too few training tokens for this context length.")
if len(val_ids) < CFG["block_size"]+2:
    print("Validation corpus too small for full context; consider lowering block_size or increasing val split.")
np.save(f"{WORK_DIR}/train_ids.npy",np.asarray(train_ids,dtype=np.uint32))
np.save(f"{WORK_DIR}/val_ids.npy",np.asarray(val_ids,dtype=np.uint32))

## Step 9 — Train safely with mixed precision, accumulation and Drive checkpoints

Safety features:
- micro-batch is 1–4 depending on detected VRAM
- gradient accumulation gives a larger effective batch without a large activation batch
- FP16 on CUDA (with GradScaler); BF16 only when supported; FP32 on CPU
- gradient clipping
- periodic checkpoint save to Drive
- resume from latest checkpoint
- detects CUDA OOM and stops cleanly with a clear instruction; reduce micro-batch/context before resuming

In [ ]:
class BlockDataset(Dataset):
    def __init__(self,arr,block): self.x=torch.from_numpy(arr.astype(np.int64)); self.block=block
    def __len__(self): return max(0,(len(self.x)-1)//self.block)
    def __getitem__(self,i):
        s=i*self.block
        return self.x[s:s+self.block],self.x[s+1:s+self.block+1]

train_ds=BlockDataset(np.load(f"{WORK_DIR}/train_ids.npy"),CFG["block_size"])
val_ds=BlockDataset(np.load(f"{WORK_DIR}/val_ids.npy"),CFG["block_size"])
if len(train_ds)==0: raise RuntimeError("No full training blocks.")
train_loader=DataLoader(train_ds,batch_size=CFG["batch_size"],shuffle=True,num_workers=0)
val_loader=DataLoader(val_ds,batch_size=1,shuffle=False,num_workers=0)

device="cuda" if torch.cuda.is_available() else "cpu"
model=JarvisGPT(CFG).to(device)
optimizer=torch.optim.AdamW(model.parameters(),lr=3e-4,weight_decay=0.1)
use_amp=(device=="cuda")
amp_dtype=torch.float16
scaler=torch.cuda.amp.GradScaler(enabled=use_amp and amp_dtype==torch.float16)

def save_checkpoint(step, best_val=None):
    path=f"{DRIVE_OUT}/checkpoints/latest.pt"
    tmp=path+".tmp"
    torch.save({"step":step,"model":model.state_dict(),"optimizer":optimizer.state_dict(),
                "config":CFG,"best_val":best_val},tmp)
    os.replace(tmp,path)
    tok.save(f"{DRIVE_OUT}/tokenizer/tokenizer.json")
    print("Checkpoint saved to Drive:",path)

def evaluate(max_batches=20):
    model.eval(); losses=[]
    with torch.no_grad():
        for i,(x,y) in enumerate(val_loader):
            if i>=max_batches: break
            x,y=x.to(device),y.to(device)
            with torch.autocast(device_type=device, dtype=amp_dtype, enabled=use_amp):
                _,loss=model(x,y)
            losses.append(float(loss.item()))
    model.train()
    return sum(losses)/max(1,len(losses))

print("Device:",device,"AMP:",use_amp,"dtype:",amp_dtype if use_amp else "fp32")
print("Train blocks:",len(train_ds),"Val blocks:",len(val_ds))

In [ ]:
# Optional resume: set RESUME=True after a runtime disconnect/restart.
RESUME=False
start_step=0
if RESUME and os.path.exists(f"{DRIVE_OUT}/checkpoints/latest.pt"):
    ck=torch.load(f"{DRIVE_OUT}/checkpoints/latest.pt",map_location=device)
    model.load_state_dict(ck["model"])
    optimizer.load_state_dict(ck["optimizer"])
    start_step=ck["step"]
    print("Resumed at step",start_step)

MAX_STEPS=1000       # smoke-test first; increase only after stable completion
ACCUM=CFG["grad_accum"]
SAVE_EVERY=100
EVAL_EVERY=100
best_val=float("inf")
model.train()
train_iter=iter(train_loader)
step=start_step
optimizer.zero_grad(set_to_none=True)

while step < MAX_STEPS:
    optimizer.zero_grad(set_to_none=True)
    running=0.0
    for micro in range(ACCUM):
        try: x,y=next(train_iter)
        except StopIteration:
            train_iter=iter(train_loader); x,y=next(train_iter)
        x,y=x.to(device),y.to(device)
        try:
            with torch.autocast(device_type=device,dtype=amp_dtype,enabled=use_amp):
                _,loss=model(x,y)
                loss=loss/ACCUM
            scaler.scale(loss).backward()
            running += float(loss.item())
        except torch.cuda.OutOfMemoryError:
            optimizer.zero_grad(set_to_none=True)
            if device=="cuda": torch.cuda.empty_cache()
            raise RuntimeError("CUDA OOM during forward/backward. Lower batch_size and/or block_size in CFG, then rerun from checkpoint.")
    scaler.unscale_(optimizer)
    torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
    scaler.step(optimizer); scaler.update()
    step+=1
    if step%25==0: print(f"step={step} train_loss={running:.4f}")
    if step%EVAL_EVERY==0:
        val=evaluate()
        print(f"step={step} val_loss={val:.4f}")
        if val<best_val: best_val=val
    if step%SAVE_EVERY==0:
        save_checkpoint(step,best_val)
save_checkpoint(step,best_val)
print("Smoke training finished. Review losses and GPU memory before increasing steps/model size.")

## Step 10 — Generate a sample

This tests the trained checkpoint. Because this model is trained from scratch on your corpus,
early responses may be incoherent. That is expected at the beginning.

In [ ]:
def generate(prompt, max_new_tokens=80, temperature=0.8, top_k=40):
    model.eval()
    prefix=f"<|bos|>\n<|user|>\n{prompt}\n<|assistant|>\n"
    ids=tok.encode(prefix).ids
    x=torch.tensor([ids[-CFG["block_size"]:]],dtype=torch.long,device=device)
    for _ in range(max_new_tokens):
        with torch.no_grad():
            with torch.autocast(device_type=device,dtype=amp_dtype,enabled=use_amp):
                logits,_=model(x)
        logits=logits[:,-1,:]/temperature
        vals,_=torch.topk(logits,min(top_k,logits.size(-1)))
        logits[logits<vals[:,-1,None]]=-float("inf")
        probs=torch.softmax(logits,dim=-1)
        nxt=torch.multinomial(probs,1)
        x=torch.cat([x,nxt],dim=1)
        if nxt.item()==tok.token_to_id("<|eos|>"): break
        if x.size(1)>CFG["block_size"]:
            x=x[:,-CFG["block_size"]:]
    decoded=tok.decode(x[0].tolist())
    return decoded.split("<|assistant|>")[-1].split("<|eos|>")[0].strip()

print(generate("Hello Jarvis, tum kya kar sakte ho?"))

## Before scaling up

Do **not** jump straight to a larger model. First confirm:
1. At least 100–500 steps finish without OOM.
2. Checkpoint exists in Drive and can be reloaded.
3. Validation loss is finite and trends downward.
4. The corpus is actually conversational and contains the target languages.
5. Colab GPU type and available VRAM are recorded.

Only then increase `MAX_STEPS`, context length, or model size one variable at a time.
For a 4 GB GPU, stay with the smallest config. For 8 GB, test the next size cautiously.
Colab runtimes can disconnect or change GPU; always save checkpoints to Drive.